# Inferencia Bayesiana: Grid → Metropolis-Hastings → Stan

Este notebook recorre tres estrategias para aproximar la distribución posterior del GLM Poisson con liga logarítmica, $\log(\mu_i) = \alpha + \beta W_i^{\text{sc}}$, y extiende el análisis al Binomial Negativo al final.

1. **Aproximación en grid:** evaluamos la posterior en una rejilla fina de 201×201 valores y normalizamos. Exacto para modelos con pocos parámetros, pero no escala: el coste crece como $n^k$ con el número de parámetros $k$.
2. **Metropolis-Hastings:** muestreamos de la posterior sin necesidad de evaluarla en todas partes, usando un paseo aleatorio en el espacio de parámetros. Escala, pero requiere calibrar el ancho de propuesta a mano.
3. **Stan (HMC/NUTS):** usa el gradiente de la log-posterior para proponer saltos informativos. Escala, se autoafina durante el warmup, y produce diagnósticos robustos ($\hat{R}$, ESS).

Los tres métodos producen posteriors consistentes para este modelo de 2 parámetros — y confirman que la aproximación normal asintótica de MLE es adecuada aquí. Al final del notebook se guardan los `InferenceData` de Poisson y Binomial Negativo en `outputs/`, listos para los PPCs en `04_predictive_checks_base_models`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf
from scipy.stats import norm

import statsmodels.graphics.tsaplots as tsp
import arviz as az
from cmdstanpy import CmdStanModel

sns.set_style('whitegrid')

## Funciones auxiliares

Las funciones de soporte se definen aquí para no interrumpir el hilo del análisis. Están agrupadas por enfoque:

- **Grid:** `get_log_likelihood`, `get_percentile_interval`, `get_hpdi`, `get_posterior_stats`
- **Metropolis-Hastings:** `plot_mcmc_diagnostics`, `plot_mcmc_joint_posterior`, `get_mcmc_stats`, `plot_parameter_comparison`
- **Stan/HMC:** `plot_stan_parameter_comparison`, `plot_density_and_normal_approx`

In [ ]:
# --- Enfoque Grid ---

def get_log_likelihood(a, b, x, y):
    eta = a + b * x
    lam = np.exp(eta)
    return np.sum(y * eta - lam)

def get_percentile_interval(grid, marginal_post, confidence=0.95):
    cdf   = np.cumsum(marginal_post)
    alpha = 1 - confidence
    idx_low  = np.where(cdf >= alpha / 2)[0][0]
    idx_high = np.where(cdf >= 1 - alpha / 2)[0][0]
    return grid[idx_low], grid[idx_high]

def get_hpdi(grid, marginal_post, confidence=0.95):
    # El HPDI es el intervalo más corto que contiene la masa de probabilidad indicada
    sorted_indices = np.argsort(marginal_post)[::-1]
    sorted_post    = marginal_post[sorted_indices]
    cdf_sorted     = np.cumsum(sorted_post)
    cutoff_idx     = np.where(cdf_sorted >= confidence)[0][0]
    hpdi_indices   = sorted_indices[:cutoff_idx + 1]
    return np.min(grid[hpdi_indices]), np.max(grid[hpdi_indices])

def get_posterior_stats(grid, marginal_post):
    p        = marginal_post / np.sum(marginal_post)
    mean     = np.sum(grid * p)
    variance = np.sum(((grid - mean)**2) * p)
    return mean, np.sqrt(variance)


# --- Enfoque Metropolis-Hastings ---

def plot_mcmc_diagnostics(trace, var_name, acceptance_rate):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].plot(trace, color='steelblue', alpha=0.7, lw=0.5)
    axes[0].set_title(f'Trace: {var_name}')
    axes[0].set_xlabel('Iteración')

    tsp.plot_acf(trace, ax=axes[1], lags=50, title=f'Autocorrelación: {var_name}')

    running_mean = np.cumsum(trace) / (np.arange(len(trace)) + 1)
    axes[2].plot(running_mean, color='steelblue')
    axes[2].set_title(f'Media acumulada: {var_name}')

    plt.suptitle(f'Diagnósticos de {var_name} — tasa de aceptación: {acceptance_rate:.1%}', fontsize=13)
    plt.tight_layout()
    plt.show()

def plot_mcmc_joint_posterior(trace_alpha, trace_beta):
    plt.figure(figsize=(8, 5))

    sns.kdeplot(x=trace_alpha, y=trace_beta, fill=True, cmap='Blues',
                thresh=0.05, levels=15, cbar=True)
    plt.scatter(trace_alpha[:500], trace_beta[:500],
                color='white', s=1, alpha=0.3, label='Muestras (subset)')

    map_alpha = np.mean(trace_alpha)
    map_beta  = np.mean(trace_beta)
    plt.plot(map_alpha, map_beta, color='tomato', marker='o',
             label=f'Media Post: α={map_alpha:.3f}, β={map_beta:.3f}')

    plt.xlabel(r'$\alpha$ (intercepto)')
    plt.ylabel(r'$\beta$ (pendiente)')
    plt.title('Densidad conjunta posterior')
    plt.legend()
    plt.tight_layout()
    plt.show()


def get_mcmc_stats(trace, confidence=0.95):
    """Estadísticas descriptivas e intervalos para una cadena MCMC."""
    mean_val = np.mean(trace)
    std_val  = np.std(trace)

    lower_p = np.percentile(trace, (1 - confidence) / 2 * 100)
    upper_p = np.percentile(trace, (1 + confidence) / 2 * 100)

    # HPDI: intervalo más corto con la masa de probabilidad requerida
    sorted_trace = np.sort(trace)
    n_samples    = len(trace)
    interval_idx = int(confidence * n_samples)
    width        = sorted_trace[interval_idx:] - sorted_trace[:n_samples - interval_idx]
    best_start   = np.argmin(width)
    hpdi_low     = sorted_trace[best_start]
    hpdi_high    = sorted_trace[best_start + interval_idx]

    return {
        'mean': mean_val, 'std': std_val,
        'p_low': lower_p, 'p_high': upper_p,
        'h_low': hpdi_low, 'h_high': hpdi_high
    }

def plot_parameter_comparison(trace, mle_mu, mle_se, var_label):
    stats   = get_mcmc_stats(trace)
    x_range = np.linspace(np.min(trace), np.max(trace), 200)

    plt.figure(figsize=(8, 5))
    sns.kdeplot(trace, label='Posterior (MH)', color='steelblue', lw=2)
    plt.plot(x_range, norm.pdf(x_range, loc=mle_mu, scale=mle_se),
             label='Aprox. normal (MLE)', color='tomato', linestyle='--')
    plt.plot(x_range, norm.pdf(x_range, loc=stats['mean'], scale=stats['std']),
             label='Aprox. normal (posterior)', color='gray', linestyle=':')
    plt.axvspan(stats['h_low'], stats['h_high'], color='steelblue', alpha=0.1, label='HPDI 95%')
    plt.title(f'Posterior de {var_label}')
    plt.xlabel('Valor del parámetro')
    plt.ylabel('Densidad')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"--- Estadísticas para {var_label} ---")
    print(f"Media: {stats['mean']:.4f} | Desv. Est: {stats['std']:.4f}")
    print(f"Intervalo Percentiles 95%: [{stats['p_low']:.4f}, {stats['p_high']:.4f}]")
    print(f"HPDI 95%:                 [{stats['h_low']:.4f}, {stats['h_high']:.4f}]")


# --- Enfoque HMC (Stan) ---

def plot_stan_parameter_comparison(fit, var_name, mle_mu, mle_se, grid_mu, grid_sigma, label):
    samples  = fit.stan_variable(var_name)
    hdi      = az.hdi(samples, hdi_prob=0.95)
    mu_stan  = np.mean(samples)
    std_stan = np.std(samples)
    x_range  = np.linspace(np.min(samples), np.max(samples), 300)

    plt.figure(figsize=(8, 5))
    sns.kdeplot(samples, label='Posterior (Stan/HMC)', color='steelblue', lw=2)
    plt.plot(x_range, norm.pdf(x_range, loc=mle_mu, scale=mle_se),
             label='Aprox. normal (MLE)', color='tomato', linestyle='--')
    plt.plot(x_range, norm.pdf(x_range, loc=grid_mu, scale=grid_sigma),
             label='Aprox. normal (grid)', color='gray', linestyle=':')
    plt.axvspan(hdi[0], hdi[1], color='steelblue', alpha=0.1, label='HDI 95% (Stan)')
    plt.title(f'Comparación de posteriors: {label}')
    plt.xlabel('Valor del parámetro')
    plt.ylabel('Densidad')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"--- Estadísticas Stan para {label} ---")
    print(f"Media: {mu_stan:.4f} | Desv. Est: {std_stan:.4f}")
    print(f"HDI 95%: [{hdi[0]:.4f}, {hdi[1]:.4f}]")

def plot_density_and_normal_approx(stan_fit, var_name):
    samples     = stan_fit.stan_variable(var_name)
    mean_sample = np.mean(samples)
    std_sample  = np.std(samples)
    x_plot      = np.linspace(np.min(samples), np.max(samples), 1000)

    plt.figure(figsize=(8, 5))
    sns.kdeplot(samples, label='Posterior (Stan/HMC)', color='steelblue', lw=2)
    plt.plot(x_plot, norm.pdf(x_plot, loc=mean_sample, scale=std_sample),
             label='Aprox. normal', color='tomato', lw=2, linestyle='--')
    plt.title(f'Posterior de {var_name}')
    plt.xlabel('Valor del parámetro')
    plt.ylabel('Densidad')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Carga de datos y referencia MLE

Cargamos los datos y ajustamos el modelo Poisson por máxima verosimilitud. Las estimaciones resultantes (`alpha_mle`, `beta_mle`) y sus errores estándar cumplen tres roles a lo largo del notebook: centran la rejilla del grid, calibran el ancho de propuesta en MH, y fijan el punto de referencia en las gráficas de comparación.

In [ ]:
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')


mean_W = df['W'].mean()
std_W = df['W'].std()

df['W_scaled'] = (df['W'] - mean_W) / std_W

x = df['W_scaled']
y = df['Sa'].values

model = smf.poisson('Sa ~ W_scaled', data=df).fit(disp=0)

alpha_mle = model.params['Intercept']
beta_mle = model.params['W_scaled']

alpha_mle_se = model.bse['Intercept']
beta_mle_se = model.bse['W_scaled']

print(f"MLE alpha: {alpha_mle:.3f} ± {alpha_mle_se:.3f}")
print(f"MLE beta:  {beta_mle:.3f} ± {beta_mle_se:.3f}")

# Enfoque Bayesiano

Hasta aquí teníamos el enfoque frecuentista: ajustamos el modelo por máxima verosimilitud y obtuvimos estimaciones puntuales con errores estándar. Lo que no tenemos es una distribución completa sobre los parámetros — solo una aproximación normal asintótica.

El enfoque bayesiano cambia la pregunta: en lugar de "¿cuál es el valor más probable de β?", pregunta "¿cuál es la distribución de β dado lo que observamos?". Esa distribución — la **posterior** — combina la verosimilitud con un **prior** que refleja lo que sabemos antes de ver los datos.

La posterior es proporcional al producto de prior y verosimilitud:

$$p(\alpha, \beta \mid y) \propto p(y \mid \alpha, \beta) \cdot p(\alpha, \beta)$$

El problema práctico es que esta distribución no tiene forma cerrada para la regresión de Poisson. Hay que aproximarla numéricamente. En este notebook exploramos tres estrategias, en orden creciente de generalidad.

---

## Aproximación en grid

La idea más directa: evaluar la posterior en una rejilla fina de valores y normalizar. Con dos parámetros y una rejilla de 201×201 puntos, calculamos la log-verosimilitud en cada combinación y exponenciamos para obtener la posterior no normalizada.

**Nota sobre el prior:** en esta implementación usamos únicamente la verosimilitud — equivale a asumir un prior uniforme (plano) sobre toda la recta real. Técnicamente esto es un prior impropio, pero en la práctica produce una posterior bien definida cuando la verosimilitud tiene un máximo claro (como aquí).

La limitación del grid es de escala: con *k* parámetros y *n* puntos por dimensión, el número de evaluaciones crece como $n^k$. Con 5 parámetros y 100 puntos, son 10 mil millones de evaluaciones. Para este modelo de 2 parámetros funciona perfectamente — pero ya con el modelo NegBin (3 parámetros) sería costoso, y con modelos jerárquicos es inviable.

### Definición del grid

La rejilla se centra en el MLE con un rango de ±5 errores estándar — suficientemente amplio para capturar la posterior aunque difiera de la aproximación normal. Con 201 puntos por dimensión, la resolución es de ~1.2 milésimas por celda en cada eje.

In [ ]:
points = 201

alpha_min = alpha_mle - 5 * alpha_mle_se
alpha_max = alpha_mle + 5 * alpha_mle_se

beta_min = beta_mle - 5 * beta_mle_se
beta_max = beta_mle + 5 * beta_mle_se

alpha_grid = np.linspace(alpha_min, alpha_max, points)
delta_alpha = alpha_grid[1] - alpha_grid[0]
beta_grid = np.linspace(beta_min, beta_max, points)
delta_beta = beta_grid[1] - beta_grid[0]

print(f"Grid de Alpha centrado en {alpha_mle:.4f} con rango [{alpha_min:.4f}, {alpha_max:.4f}]")
print(f"Grid de Beta centrado en {beta_mle:.4f} con rango [{beta_min:.4f}, {beta_max:.4f}]")

### Cálculo de la posterior y las marginales

Evaluamos la log-verosimilitud en cada punto del grid. Antes de exponenciar restamos el máximo (estabilidad numérica) y normalizamos la suma a 1. Las posteriors marginales se obtienen sumando sobre la dimensión que no nos interesa — el equivalente discreto de integrar fuera el parámetro nuisance.

In [ ]:
A, B = np.meshgrid(alpha_grid, beta_grid)

log_likelihood_surface = np.zeros((points, points))

for i in range(points):
    for j in range(points):
        log_likelihood_surface[i, j] = get_log_likelihood(alpha_grid[j], beta_grid[i], x, y)

log_l_max = np.max(log_likelihood_surface)
posterior_unnorm = np.exp(log_likelihood_surface - log_l_max)
posterior = posterior_unnorm / np.sum(posterior_unnorm)

posterior_alpha = np.sum(posterior, axis=0)
posterior_beta = np.sum(posterior, axis=1)

### Visualizaciones

Tres gráficas complementarias: el mapa de calor de la posterior conjunta (distribución de masa en el espacio $(\alpha, \beta)$), y las posteriors marginales de cada parámetro comparadas con la aproximación normal de MLE. Si el MLE aproxima bien la posterior, las curvas deben solaparse — resultado que confirmaremos con MH y Stan.

In [ ]:
plt.figure(figsize=(8, 5))
plt.contourf(A, B, posterior, cmap='Blues', levels=20)
plt.colorbar(label='Densidad posterior')
plt.xlabel(r'$\alpha$ (intercepto)')
plt.ylabel(r'$\beta$ (pendiente)')
plt.title('Posterior conjunta — aproximación en grid')

idx = np.unravel_index(np.argmax(posterior), posterior.shape)
print(f"MAP estimado: alpha = {alpha_grid[idx[1]]:.3f}, beta = {beta_grid[idx[0]]:.3f}")
plt.plot(alpha_grid[idx[1]], beta_grid[idx[0]], color='tomato', marker='o', label='MAP')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
alpha_low_p, alpha_high_p = get_percentile_interval(alpha_grid, posterior_alpha, 0.95)
alpha_low_h, alpha_high_h = get_hpdi(alpha_grid, posterior_alpha, 0.95)
mu_alpha, sigma_alpha = get_posterior_stats(alpha_grid, posterior_alpha)

densidad_mle   = norm.pdf(alpha_grid, loc=alpha_mle, scale=alpha_mle_se)
densidad_bayes = norm.pdf(alpha_grid, loc=mu_alpha, scale=sigma_alpha)

plt.figure(figsize=(8, 5))
sns.lineplot(x=alpha_grid, y=posterior_alpha / delta_alpha,
             label='Posterior (grid)', color='steelblue')
sns.lineplot(x=alpha_grid, y=densidad_mle,
             label='Aprox. normal (MLE)', linestyle='--', color='tomato')
sns.lineplot(x=alpha_grid, y=densidad_bayes,
             label='Aprox. normal (posterior)', linestyle=':', color='gray')
plt.title(r'Posterior marginal de $\alpha$')
plt.xlabel(r'$\alpha$')
plt.ylabel('Densidad')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Intervalo Percentiles 95%: [{alpha_low_p:.3f}, {alpha_high_p:.3f}]")
print(f"Intervalo HPDI 95%:        [{alpha_low_h:.3f}, {alpha_high_h:.3f}]")
print(f"alpha: {mu_alpha:.4f} ± {sigma_alpha:.4f}")

In [ ]:
beta_low_p, beta_high_p = get_percentile_interval(beta_grid, posterior_beta, 0.95)
beta_low_h, beta_high_h = get_hpdi(beta_grid, posterior_beta, 0.95)
mu_beta, sigma_beta = get_posterior_stats(beta_grid, posterior_beta)

densidad_mle   = norm.pdf(beta_grid, loc=beta_mle, scale=beta_mle_se)
densidad_bayes = norm.pdf(beta_grid, loc=mu_beta, scale=sigma_beta)

plt.figure(figsize=(8, 5))
sns.lineplot(x=beta_grid, y=posterior_beta / delta_beta,
             label='Posterior (grid)', color='steelblue')
sns.lineplot(x=beta_grid, y=densidad_mle,
             label='Aprox. normal (MLE)', linestyle='--', color='tomato')
sns.lineplot(x=beta_grid, y=densidad_bayes,
             label='Aprox. normal (posterior)', linestyle=':', color='gray')
plt.title(r'Posterior marginal de $\beta$')
plt.xlabel(r'$\beta$')
plt.ylabel('Densidad')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Intervalo Percentiles 95%: [{beta_low_p:.3f}, {beta_high_p:.3f}]")
print(f"Intervalo HPDI 95%:        [{beta_low_h:.3f}, {beta_high_h:.3f}]")
print(f"beta: {mu_beta:.4f} ± {sigma_beta:.4f}")

# Metropolis-Hastings

El grid confirmó que la posterior de Poisson es bien comportada y muy cercana a la aproximación normal de MLE. Pero la limitación que señalamos es real: no escala. El modelo NegBin tiene un parámetro extra de sobredispersión φ, y cualquier modelo jerárquico tendría decenas.

La alternativa es **no evaluar la posterior en toda la rejilla**, sino *muestrear* de ella. Si tenemos suficientes muestras, podemos estimar cualquier estadístico de la posterior sin necesidad de conocerla en todas partes.

Metropolis-Hastings es el algoritmo MCMC más simple que hace esto. La idea es caminar por el espacio de parámetros: en cada paso se propone un movimiento y se acepta con probabilidad proporcional al cociente de posteriors. Formalmente, si la posterior actual es $p(\theta)$ y la propuesta es $p(\theta')$:

$$\alpha = \min\left(1,\ \frac{p(\theta' \mid y)}{p(\theta \mid y)}\right)$$

Como usamos log-verosimilitudes (para evitar underflow), la condición de aceptación es:

$$\log u < \log p(\theta' \mid y) - \log p(\theta \mid y), \quad u \sim \text{Uniforme}(0,1)$$

La propuesta es una normal centrada en el estado actual — un "paseo aleatorio" en el espacio de parámetros. El ancho de la propuesta controla la tasa de aceptación: demasiado angosto y la cadena no explora, demasiado ancho y casi todo se rechaza. La tasa óptima teórica es ~23-45% para distribuciones suaves.

**Nota sobre el prior:** igual que en el grid, el código solo usa la log-verosimilitud en el cociente de aceptación. El prior implícito sigue siendo uniforme — la posterior muestreada es proporcional a la verosimilitud.

In [ ]:
n_iterations = 20000
burn_in = 5000

alpha_curr = alpha_mle 
beta_curr = beta_mle

trace_alpha = np.zeros(n_iterations)
trace_beta = np.zeros(n_iterations)
accepted = 0

proposal_width_a = alpha_mle_se * 2
proposal_width_b = beta_mle_se * 2

for i in range(n_iterations):

    alpha_prop = np.random.normal(alpha_curr, proposal_width_a)
    beta_prop = np.random.normal(beta_curr, proposal_width_b)
    
    lp_curr = get_log_likelihood(alpha_curr, beta_curr, x, y)
    lp_prop = get_log_likelihood(alpha_prop, beta_prop, x, y)
    
    log_r = lp_prop - lp_curr
    
    if np.log(np.random.uniform(0, 1)) < log_r:
        alpha_curr = alpha_prop
        beta_curr = beta_prop
        accepted += 1
        
    trace_alpha[i] = alpha_curr
    trace_beta[i] = beta_curr

trace_alpha = trace_alpha[burn_in:]
trace_beta = trace_beta[burn_in:]
acceptance_rate = accepted / n_iterations

print(f"Tasa de aceptación: {acceptance_rate:.2%}")
print(f"Alpha (Posterior Mean): {np.mean(trace_alpha):.4f}")
print(f"Beta  (Posterior Mean): {np.mean(trace_beta):.4f}")

### Diagnósticos de convergencia

Para confiar en las muestras de MH verificamos tres cosas: que la cadena exploró bien el espacio (trace plot), que las muestras sucesivas no están demasiado correlacionadas (autocorrelación), y que la media empírica estabilizó (media acumulada). Una tasa de aceptación entre 23% y 45% indica que el ancho de propuesta está bien calibrado.

In [ ]:
plot_mcmc_diagnostics(trace_alpha, 'Alpha (Intercepto)', acceptance_rate)
plot_mcmc_diagnostics(trace_beta, 'Beta (Ancho)', acceptance_rate)

### Distribuciones posteriores

Con la cadena convergente, visualizamos la posterior conjunta en 2D y comparamos cada marginal con la aproximación normal de MLE. La correspondencia entre ambas confirma que la posterior del GLM Poisson es bien comportada: la aproximación asintótica de MLE es suficiente para este modelo de dos parámetros.

In [ ]:
plot_mcmc_joint_posterior(trace_alpha, trace_beta)

In [ ]:
plot_parameter_comparison(trace_alpha, alpha_mle, alpha_mle_se, r"$\alpha$ (Intercepto)")
plot_parameter_comparison(trace_beta, beta_mle, beta_mle_se, r"$\beta$ (Pendiente)")

# Stan (HMC)

Metropolis-Hastings con propuesta de paseo aleatorio funciona, pero tiene dos limitaciones prácticas:

1. **Afinación manual:** el ancho de la propuesta hay que ajustarlo a mano para cada modelo. Lo que funciona para Poisson no funciona necesariamente para NegBin o para un modelo jerárquico.
2. **Exploración ineficiente en espacios de alta dimensión:** el paseo aleatorio tarda mucho en atravesar una distribución alargada o correlacionada — necesita muchos más pasos que un algoritmo que usa información del gradiente.

**Hamiltonian Monte Carlo (HMC)** resuelve ambos problemas usando el gradiente de la log-posterior para proponer saltos informativos (análogos a la dinámica de una partícula en un campo de energía). Stan implementa una variante adaptativa, **NUTS** (No-U-Turn Sampler), que además ajusta automáticamente los hiperparámetros durante el warmup.

En la práctica: le damos a Stan el modelo en su lenguaje probabilístico, los datos, y él se encarga del resto. Las cadenas convergen más rápido, tienen menor autocorrelación, y el diagnóstico es más robusto (R̂, ESS).

La comparación de los tres métodos al final de este notebook muestra que las posteriors son consistentes entre sí — lo que esperaríamos si todos implementan el mismo modelo bajo el mismo prior (uniforme implícito en Grid y MH, explícito en Stan con priors difusos).

In [ ]:
stan_data = {
    'N': len(df),
    'x': df['W_scaled'].values,
    'y': df['Sa'].values
}

model_stan = CmdStanModel(stan_file='../models/poisson_model.stan')
fit = model_stan.sample(data=stan_data, chains=4, iter_sampling=2000)

In [ ]:
display(fit.summary())

alpha_samples = fit.stan_variable('alpha')
beta_samples = fit.stan_variable('beta')

az.plot_trace(fit, var_names=['alpha', 'beta'], compact=False)
plt.tight_layout()
plt.show()

# 4. Gráfica de Autocorrelación (Stan suele tener mucha menos que MH)
az.plot_autocorr(fit, var_names=['alpha', 'beta'], combined=True)
plt.show()

In [ ]:
plot_mcmc_joint_posterior(alpha_samples, beta_samples)

In [ ]:
plot_stan_parameter_comparison(fit, 'alpha', alpha_mle, alpha_mle_se, mu_alpha, sigma_alpha, r"$\alpha$ (Intercepto)")
plot_stan_parameter_comparison(fit, 'beta', beta_mle, beta_mle_se, mu_beta, sigma_beta, r"$\beta$ (Ancho)")

## Guardado de resultados

Convertimos el fit de Stan al formato `az.InferenceData` incluyendo tres componentes que Stan ya generó durante el muestreo:

- `posterior_predictive` (`y_rep`): réplicas de los datos bajo la distribución predictiva posterior — necesarias para los PPCs.
- `log_likelihood` (`log_lik`): log-verosimilitudes punto a punto — necesarias para el cálculo de LOO-CV en el notebook `05_predictive_checks_extended_models`.

Los archivos se guardan en `outputs/` (excluido de git) y se cargan directamente en los notebooks 04 y 05.

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

idata_poisson = az.from_cmdstanpy(
    fit,
    observed_data={'y': stan_data['y']},
    posterior_predictive='y_rep',
    log_likelihood='log_lik'
)
az.to_netcdf(idata_poisson, '../outputs/pois_idata.nc')
print("Guardado: ../outputs/pois_idata.nc")

## Modelo Binomial Negativo

El diagnóstico de sobredispersión en `01_basic_eda` motivó el Binomial Negativo como extensión del Poisson. Aquí ajustamos la versión bayesiana con Stan para obtener muestras posteriores de sus tres parámetros: `alpha` (intercepto), `beta` (pendiente), y `alpha_sm` (parámetro de sobredispersión $\phi$ — el sufijo `_sm` evita colisión de nombres con el intercepto `alpha` en el modelo Stan).

El objetivo no es analizar este modelo en detalle — eso ocurre en `04_predictive_checks_base_models`. Aquí solo necesitamos las muestras posteriores guardadas en `outputs/neg_idata.nc`.

In [ ]:
stan_data = {
    'N': len(df),
    'x': df['W_scaled'].values,
    'y': df['Sa'].values
}

def get_inits():
    return {
        "alpha": 1.0, 
        "beta": 0.4, 
        "phi": 1.0
    }

model_nb = CmdStanModel(stan_file='../models/negative_binomial_model.stan')
fit_nb = model_nb.sample(data=stan_data, chains=4, iter_sampling=2000, inits=get_inits())

In [ ]:
display(fit_nb.summary())

az.plot_trace(fit_nb, var_names=['alpha', 'beta','alpha_sm'], compact=False)
plt.tight_layout()
plt.show()


# 4. Gráfica de Autocorrelación (Stan suele tener mucha menos que MH)
az.plot_autocorr(fit_nb, var_names=['alpha', 'beta','alpha_sm'], combined=True)
plt.show()

In [ ]:
alpha_nb_samples    = fit_nb.stan_variable('alpha')
beta_nb_samples     = fit_nb.stan_variable('beta')
# alpha_sm: parámetro de sobredispersión φ (el sufijo _sm evita colisión con el intercepto alpha)
alpha_sm_nb_samples = fit_nb.stan_variable('alpha_sm')

mean_alpha_nb    = np.mean(alpha_nb_samples)
std_alpha_nb     = np.std(alpha_nb_samples)

mean_beta_nb     = np.mean(beta_nb_samples)
std_beta_nb      = np.std(beta_nb_samples)

mean_alpha_sm_nb = np.mean(alpha_sm_nb_samples)
std_alpha_sm_nb  = np.std(alpha_sm_nb_samples)

In [ ]:
plot_mcmc_joint_posterior(alpha_nb_samples, beta_nb_samples)

In [ ]:
plot_density_and_normal_approx(fit_nb, var_name='alpha')
plot_density_and_normal_approx(fit_nb, var_name='beta')
plot_density_and_normal_approx(fit_nb, var_name='alpha_sm')

In [ ]:
idata_nb = az.from_cmdstanpy(
    fit_nb,
    observed_data={'y': stan_data['y']},
    posterior_predictive='y_rep',
    log_likelihood='log_lik'
)
az.to_netcdf(idata_nb, '../outputs/neg_idata.nc')
print("Guardado: ../outputs/neg_idata.nc")

---

## ¿Qué sigue?

Con las muestras posteriores de Poisson y Binomial Negativo tenemos distribuciones completas sobre los parámetros. Pero eso responde "¿cuánto vale β?" — no "¿describe bien este modelo los datos observados?"

La siguiente etapa del workflow bayesiano son los **Posterior Predictive Checks (PPCs)**: simulamos datos nuevos desde la distribución predictiva posterior y los comparamos con los datos reales en estadísticos relevantes (media, varianza, proporción de ceros, máximo).

El notebook `04_predictive_checks_base_models` implementa este diagnóstico para Poisson y Binomial Negativo.